# Chapter 3. Basic Ray Tracing

The code from Chapter 3 of *Radio Propagation Ray Tracing with Sionna RT*.
The cells follow the order of the text, and the contents of each code cell are
the same as printed there.

Only the first Setup cell is not printed in the book. It performs the imports
and the scene loading that the printed code assumes. Once the setup cell has
run, the rest can be run from the top in order.

Fragments the text shows for explanation, such as a path to a scene of your own
or an object name that scene does not contain, will not run as they stand. A
note appears before each such cell.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

## 1. Loading a First Scene

In [ ]:
from sionna.rt import load_scene, scene as scene_assets

scene = load_scene(scene_assets.simple_street_canyon)
print(f'objects: {len(scene.objects)}')

In [ ]:
scene.frequency = 3.5e9   # 3.5 GHz, representative of Sub-6 GHz

## 2. Placing a Transmitter

In [ ]:
from sionna.rt import Transmitter, Receiver

tx = Transmitter(name='tx0',
                 position=[-70.0, -6.0, 10.0])  # units: meters
scene.add(tx)

## 3. Placing a Receiver

In [ ]:
rx = Receiver(name='rx0',
              position=[70.0, 6.0, 1.5])
scene.add(rx)

## 4. Configuring the Antennas

In [ ]:
from sionna.rt import PlanarArray

# Tx side: single element, vertical polarization, isotropic pattern
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern='iso',
                             polarization='V')

# Same configuration on the Rx side
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern='iso',
                             polarization='V')

## 5. Computing Propagation Paths with PathSolver

In [ ]:
from sionna.rt import PathSolver

solver = PathSolver()
paths = solver(scene, max_depth=3,
               los=True, specular_reflection=True,
               diffraction=False, diffuse_reflection=False)

# shape: (rx, tx, number of paths)
print('valid:', paths.valid.shape)

## 6. Checking the LoS Path

In [ ]:
import numpy as np

# max_depth=0 computes the LoS path only
paths_los = solver(scene, max_depth=0, los=True,
                   specular_reflection=False)
print('LoS paths:', int(np.array(paths_los.valid).sum()))
a_los = paths_los.a   # (real, imag) tuple
print('|a| (LoS):', np.sqrt(a_los[0]**2 + a_los[1]**2))

## 7. Checking the Reflected Paths

In [ ]:
import numpy as np

paths_1 = solver(scene, max_depth=1, los=True,
                 specular_reflection=True)
paths_3 = solver(scene, max_depth=3, los=True,
                 specular_reflection=True)

print('max_depth=1 paths:', int(np.array(paths_1.valid).sum()))
print('max_depth=3 paths:', int(np.array(paths_3.valid).sum()))

## 9. Displaying the Paths

In [ ]:
v = np.array(paths.vertices)         # coordinates of the interaction points
kind = np.array(paths.interactions)  # interaction type (0 means none)

In [ ]:
from sionna.rt import Camera
cam = Camera(position=[-180, -180, 190], look_at=[0, 0, 0])
fig = scene.render(camera=cam, paths=paths, clip_at=13.0)

## 10. The Relation Between Received Power and Distance

In [ ]:
import numpy as np
# for this experiment only, move the Tx to the end of the road
tx_sweep = np.array([-90.0, -6.0, 10.0])
for dx in np.linspace(20, 160, 22):  # separation along x [m]
    scene.remove('rx0')
    rx_pos = np.array([-90.0 + dx, 6.0, 1.5])
    distance = np.linalg.norm(rx_pos - tx_sweep)  # 3D distance
    scene.add(Receiver('rx0', position=rx_pos.tolist()))
    a = solver(scene, max_depth=3).a   # (real, imag) tuple
    # add 30 dBm to the summed path gain
    prx_dbm = 10*np.log10((a[0]**2 + a[1]**2).sum()) + 30

In [ ]:
for depth in [0, 1, 2, 3, 5]:
    paths = solver(scene, max_depth=depth)
    a = paths.a   # (real, imag) tuple
    prx_dbm = 10*np.log10((a[0]**2 + a[1]**2).sum().item()) + 30
    print(f'max_depth={depth}: P_rx = {prx_dbm:.2f} dBm')